In [1]:
import pandas as pd
import numpy as np
import requests
from io import StringIO
import re
import json

In [3]:
url = "https://services.swpc.noaa.gov/json/rtsw/rtsw_wind_1m.json"
#url = "https://services.swpc.noaa.gov/json/rtsw/rtsw_mag_1m.json"
data = requests.get(url).json()
df = pd.DataFrame(data)

for i in range(20):
    print(data[i])
df
# for i in range(20):
#     print(data[i+1000])

{'time_tag': '2026-08-04T16:06:00', 'active': True, 'source': 'SOLAR1', 'proton_speed': 447.4, 'proton_temperature': 59286, 'proton_density': 2.18, 'proton_vx_gse': -447.1, 'proton_vy_gse': 11.5, 'proton_vz_gse': -10.4, 'proton_vx_gsm': -447.1, 'proton_vy_gsm': 7.6, 'proton_vz_gsm': -13.5, 'proton_sample_size': 1, 'alpha_speed': None, 'alpha_temperature': None, 'alpha_density': None, 'alpha_vx_gse': None, 'alpha_vy_gse': None, 'alpha_vz_gse': None, 'alpha_vx_gsm': None, 'alpha_vy_gsm': None, 'alpha_vz_gsm': None, 'alpha_sample_size': None, 'max_convergence_flag': 0, 'max_data_flag': 0, 'max_error_count_flag': 0, 'max_processing_flag': 0, 'max_range_flag': 0, 'max_sample_count_flag': 0, 'max_telemetry_flag': 0, 'overall_quality': 0}
{'time_tag': '2026-08-04T16:05:08', 'active': False, 'source': 'IMAP', 'proton_speed': 463.21, 'proton_temperature': 49104, 'proton_density': 2.6, 'proton_vx_gse': None, 'proton_vy_gse': None, 'proton_vz_gse': None, 'proton_vx_gsm': None, 'proton_vy_gsm': No

,time_tag,active,source,proton_speed,proton_temperature,proton_density,proton_vx_gse,proton_vy_gse,proton_vz_gse,proton_vx_gsm,...,alpha_vz_gsm,alpha_sample_size,max_convergence_flag,max_data_flag,max_error_count_flag,max_processing_flag,max_range_flag,max_sample_count_flag,max_telemetry_flag,overall_quality
0,2026-08-04T16:06:00,True,SOLAR1,447.40,59286,2.18,-447.1,11.5,-10.4,-447.1,...,None,None,0,0,0,0,0,0,0,0
1,2026-08-04T16:05:08,False,IMAP,463.21,49104,2.60,NaN,NaN,NaN,NaN,...,None,None,0,0,0,0,0,0,0,0
2,2026-08-04T16:05:00,True,SOLAR1,453.30,56864,2.31,-452.6,9.4,-23.0,-452.6,...,None,None,0,0,0,0,0,0,0,0
3,2026-08-04T16:05:00,False,ACE,451.10,50990,2.19,NaN,NaN,NaN,NaN,...,None,None,0,0,0,0,0,0,0,0
4,2026-08-04T16:04:08,False,IMAP,465.50,46102,2.54,NaN,NaN,NaN,NaN,...,None,None,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3566,2026-08-03T16:11:00,True,SOLAR1,349.20,237653,10.36,-348.2,14.7,-21.8,-348.2,...,None,None,0,0,0,0,0,0,0,0
3567,2026-08-03T16:11:00,False,ACE,378.46,151573,4.12,NaN,NaN,NaN,NaN,...,None,None,0,0,0,0,0,0,0,0
3568,2026-08-03T16:10:08,False,IMAP,345.70,316649,26.72,NaN,NaN,NaN,NaN,...,None,None,0,0,0,0,0,0,0,0
3569,2026-08-03T16:10:00,True,SOLAR1,346.20,221556,10.10,-344.9,14.7,-26.5,-344.9,...,None,None,0,0,0,0,0,0,0,0


There seem to be two types of sources/satellites that NOAA pulls from: ACE and SOLAR1\. Only one of them is active at each timestamp\. That's the subset of data we will use for our real\-time dataset\.

In [5]:
df[df["active"]]

,time_tag,active,source,proton_speed,proton_temperature,proton_density,proton_vx_gse,proton_vy_gse,proton_vz_gse,proton_vx_gsm,...,alpha_vz_gsm,alpha_sample_size,max_convergence_flag,max_data_flag,max_error_count_flag,max_processing_flag,max_range_flag,max_sample_count_flag,max_telemetry_flag,overall_quality
0,2026-08-04T16:06:00,True,SOLAR1,447.4,59286,2.18,-447.1,11.5,-10.4,-447.1,...,None,None,0,0,0,0,0,0,0,0
2,2026-08-04T16:05:00,True,SOLAR1,453.3,56864,2.31,-452.6,9.4,-23.0,-452.6,...,None,None,0,0,0,0,0,0,0,0
5,2026-08-04T16:04:00,True,SOLAR1,450.4,60972,2.25,-450.1,7.9,-13.4,-450.1,...,None,None,0,0,0,0,0,0,0,0
8,2026-08-04T16:03:00,True,SOLAR1,457.3,77156,2.38,-457.1,10.0,-11.0,-457.1,...,None,None,0,0,0,0,0,0,0,0
12,2026-08-04T16:02:00,True,SOLAR1,463.9,71831,2.62,-463.5,7.7,-16.5,-463.5,...,None,None,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3558,2026-08-03T16:14:00,True,SOLAR1,353.5,256009,11.85,-352.7,14.3,-19.0,-352.7,...,None,None,0,0,0,0,0,0,0,0
3560,2026-08-03T16:13:00,True,SOLAR1,355.6,240341,11.51,-354.4,19.2,-21.5,-354.4,...,None,None,0,0,0,0,0,0,0,0
3563,2026-08-03T16:12:00,True,SOLAR1,352.3,243618,11.41,-351.1,18.0,-22.7,-351.1,...,None,None,0,0,0,0,0,0,0,0
3566,2026-08-03T16:11:00,True,SOLAR1,349.2,237653,10.36,-348.2,14.7,-21.8,-348.2,...,None,None,0,0,0,0,0,0,0,0


In [7]:
active_df = df[df["active"]].copy()
active_df["time_tag"].is_unique

False

In [9]:
def _prepare_dataframe(df, numeric_cols, interval=180):

    # Convert datatypes
    df["time_tag"] = pd.to_datetime(df["time_tag"])

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df[numeric_cols] = df[numeric_cols].replace(-9999, np.nan)

    latest = df["time_tag"].max()

    full_index = pd.date_range(
        end=latest,
        periods=interval,
        freq="1min"
    )

    df = (
        df.set_index("time_tag")
          .reindex(full_index)
          .rename_axis("time_tag")
          .reset_index()
    )

    return df

In [17]:
def get_mag_data():
    url = "https://services.swpc.noaa.gov/json/rtsw/rtsw_mag_1m.json"
    df = pd.DataFrame(requests.get(url).json())
    df["active"] = df["active"].astype(bool)
    df = (
        df[df["active"]]
        [["time_tag", "bt", "bx_gsm", "by_gsm", "bz_gsm"]]
    )
    return df

def get_proton_data():
    url = "https://services.swpc.noaa.gov/json/rtsw/rtsw_wind_1m.json"
    df = pd.DataFrame(requests.get(url).json())
    df["active"] = df["active"].astype(bool)
    df = (
        df[df["active"]]
        [["time_tag", "proton_speed", "proton_density"]]
    )
    return df

def get_real_time_data(interval=180):

    mag_df = get_mag_data()
    proton_df = get_proton_data()

    merged_df = pd.merge(
        mag_df,
        proton_df,
        on="time_tag",
        how="outer"
    )

    merged_df = _prepare_dataframe(
        merged_df,
        [
            "bt",
            "bx_gsm",
            "by_gsm",
            "bz_gsm",
            "proton_speed",
            "proton_density"
        ],
        interval
    )

    merged_df = merged_df.rename(columns={
        "time_tag": "timestamp",
        "bt": "mag_avg_nt",
        "bx_gsm": "bx_gsm_nt",
        "by_gsm": "by_gsm_nt",
        "bz_gsm": "bz_gsm_nt",
        "proton_speed": "flow_speed_km_s",
        "proton_density": "proton_density_n_cc"
    })

    return merged_df

df = get_real_time_data()
df

In [53]:
# Missing a couple of timestamps, we can impute them the same way we did with our training data
df.isnull().sum()

timestamp              0
mag_avg_nt             0
bx_gsm_nt              0
by_gsm_nt              0
bz_gsm_nt              0
flow_speed_km_s        2
proton_density_n_cc    2
dtype: int64

In [13]:
for feature in df.columns:
    df[feature] = df[feature].interpolate(
        method="linear",
    )

/tmp/ipykernel_1213/1387747268.py:2: FutureWarning: Series.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df[feature] = df[feature].interpolate(


In [15]:
df.describe()

,proton_speed,proton_temperature,proton_density,proton_vx_gse,proton_vy_gse,proton_vz_gse,proton_vx_gsm,proton_vy_gsm,proton_vz_gsm,proton_sample_size,max_convergence_flag,max_data_flag,max_error_count_flag,max_processing_flag,max_range_flag,max_sample_count_flag,max_telemetry_flag,overall_quality
count,3571.000000,3571.000000,3571.000000,3571.000000,3571.000000,3571.000000,3571.000000,3571.000000,3571.000000,3571.0,3571.0,3571.0,3571.0,3571.0,3571.0,3571.0,3571.0,3571.0
mean,247.651112,123276.822739,-132.221694,-799.814590,-399.657141,-427.284052,-799.814590,-400.176771,-435.980412,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
std,1209.120231,75319.198137,1163.976802,1720.837947,1795.639583,1790.449783,1720.837947,1795.473321,1788.957153,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
min,-9999.000000,-9999.000000,-9999.000000,-9999.000000,-9999.000000,-9999.000000,-9999.000000,-9999.000000,-9999.000000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25%,365.900000,61320.000000,2.120000,-414.150000,11.600000,-13.533333,-414.150000,7.216667,-20.000000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
50%,383.700000,101275.000000,2.670000,-377.133333,27.300000,1.400000,-377.133333,26.333333,-6.700000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
75%,412.810000,176498.000000,6.150000,-360.300000,49.966667,18.800000,-360.300000,52.633333,7.841667,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
max,514.680000,435297.000000,122.820000,-335.900000,138.100000,85.100000,-335.900000,137.900000,81.700000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a18bf7d-431c-4909-af8a-a54a62228a78' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>